[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_17/MFM_ch17_seminar/MFM_ch17_seminar.ipynb)

# MFM_ch17_seminar

Seminar 17 computations: present value and rational bubbles, the Blanchard-Watson bubble, the Dickey-Fuller limit and a mildly explosive root by simulation, Markov-switching filter steps, LPPLS diagnostics; GSADF on the Shiller price-dividend ratio with three minimum windows, SADF vs GSADF, Bitcoin with Monte Carlo vs wild bootstrap critical values, BET and BET-FI, LPPLS critical-time dispersion, the Nasdaq 100 confidence indicator, Markov-switching for the S&P 500 and Bitcoin, and the AI-bubble question for NVIDIA and the Nasdaq 100; charts for every task and solution: the seminar data, the bubble share, a simulated Evans bubble, the Dickey-Fuller and mildly explosive distributions, filter steps, LPPLS trends and drifts, BSADF for three minimum windows, PWY vs PSY for the S&P 500, and the size of GSADF under changing volatility.

*Modelling Financial Markets (MFM), Chapter 17: Bubbles and Crashes. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_17'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# Colab: !pip install xlrd
import io
import json
import os
import re
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import optimize, stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'   # doar linii de referinta, benzi, grila
Teal     = '#17A2B8'
Magenta  = '#D63384'
Brown    = '#795548'

HERE = '.'
SEED = 42
R_MC = 2000
RECOMPUTE = False   # True: recalculeaza indicatorii LPPLS (cateva minute); False: ii citeste din fisierele publicate
QL_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/Ch_17/'


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def fig_legend(fig, axes, ncol=3, y=0.0):
    """O singura legenda pentru o figura cu mai multe panouri, sub figura."""
    h, l = [], []
    for ax in np.atleast_1d(axes):
        for hh, ll in zip(*ax.get_legend_handles_labels()):
            if ll not in l and not ll.startswith('_'):
                h.append(hh)
                l.append(ll)
    fig.legend(h, l, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer, np.bool_)):
        return o.item()
    if isinstance(o, pd.Timestamp):
        return str(o.date())
    return o


def yrs(idx):
    """Data -> ani zecimali."""
    idx = pd.DatetimeIndex(idx)
    return np.asarray(idx.year + (idx.dayofyear - 1) / 365.25)


def todate(x):
    """Ani zecimali -> data."""
    y = int(np.floor(x))
    return pd.Timestamp(f'{y}-01-01') + pd.Timedelta(days=float((x - y) * 365.25))


def d2s(d):
    return None if d is None else str(pd.Timestamp(d).date())

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# nume -> (simbol, eticheta, tip, coloana, data de start)
MARKETS = {
    'sp500': ('GSPC.INDX',   'S&P 500',        'index',  'close',          '1990-01-01'),
    'ndx':   ('NDX.INDX',    'Nasdaq 100',     'index',  'close',          '1990-01-01'),
    'bet':   ('BET',         'BET',            'index',  'close',          '1997-09-19'),
    'betfi': ('BETFI.INDX',  'BET-FI',         'index',  'close',          '2012-01-25'),
    'btc':   ('BTC-USD.CC',  'Bitcoin',        'crypto', 'close',          '2014-09-17'),
    'eth':   ('ETH-USD.CC',  'Ethereum',       'crypto', 'close',          '2016-01-01'),
    'doge':  ('DOGE-USD.CC', 'Dogecoin',       'crypto', 'close',          '2017-01-01'),
    'nvda':  ('NVDA.US',     'NVIDIA',         'stock',  'adjusted_close', '1999-01-22'),
    'csco':  ('CSCO.US',     'Cisco Systems',  'stock',  'adjusted_close', '1990-02-16'),
    'tsla':  ('TSLA.US',     'Tesla',          'stock',  'adjusted_close', '2010-06-29'),
    'gme':   ('GME.US',      'GameStop',       'stock',  'adjusted_close', '2002-02-13'),
    'mstr':  ('MSTR.US',     'Strategy (MicroStrategy)', 'stock', 'adjusted_close', '1998-06-11'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

SHILLER_URLS = ['https://img1.wsimg.com/blobby/go/e5e77e0b-59d1-44d9-ab25-4763ac982e53/downloads/ie_data.xls',
                'http://www.econ.yale.edu/~shiller/data/ie_data.xls']
FRENCH = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def price(name, freq='D', start=None, end=END):
    """Nivelul seriei: 'D' zilnic (curatat), 'W' ultimul pret al saptamanii, 'M' ultimul pret al lunii."""
    symbol, _, kind, col, start0 = MARKETS[name]
    s = read_market(symbol)[col].loc[start or start0:end]
    s = s[s > 0].dropna()
    if kind != 'crypto':
        s = s[s.index.dayofweek < 5]          # fara cotatii de weekend
    if kind == 'index':
        s = s[s.diff() != 0]                  # fara sarbatori completate cu pretul anterior
    last = s.index[-1]
    if freq == 'W':
        s = s.resample('W-FRI').last().dropna()
    elif freq == 'M':
        s = s.resample('ME').last().dropna()
    if freq in ('W', 'M') and s.index[-1] > last:
        s.index = s.index[:-1].append(pd.DatetimeIndex([last]))   # ultima perioada incompleta: datata la ultima observatie
    return s.rename(name)


def shiller_urls():
    """Adresele fisierului ie_data.xls: legatura curenta de pe shillerdata.com, apoi copiile cunoscute."""
    urls = []
    try:
        html = urllib.request.urlopen(urllib.request.Request('https://shillerdata.com/', headers={'User-Agent': 'Mozilla/5.0'}),
                                      timeout=60).read().decode('utf-8', 'ignore')
        urls += ['https:' + u if u.startswith('//') else u
                 for u in re.findall(r'href="([^"]*ie_data\.xls[^"]*)"', html)]
    except Exception:
        pass
    return urls + SHILLER_URLS


def shiller():
    """S&P Composite lunar (Shiller): pret real, dividend real si raportul pret/dividend (P/D)."""
    if 'shiller' in _CACHE:
        return _CACHE['shiller']
    raw = None
    for url in shiller_urls():
        try:
            raw = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'}),
                                         timeout=120).read()
            break
        except Exception:
            continue
    d = pd.read_excel(io.BytesIO(raw), sheet_name='Data', header=None, skiprows=8)
    d = d.iloc[:, [0, 1, 2, 7, 8]]
    d.columns = ['date', 'P', 'D', 'real_P', 'real_D']
    d = d[pd.to_numeric(d['date'], errors='coerce').notna()].copy()
    yr = d['date'].astype(float)
    year = np.floor(yr + 1e-9).astype(int)
    month = np.round((yr - year) * 100).astype(int)
    d.index = pd.to_datetime(dict(year=year, month=month, day=1)) + pd.offsets.MonthEnd(0)
    d = d[['P', 'D', 'real_P', 'real_D']].apply(pd.to_numeric, errors='coerce').dropna()
    d['PD'] = d['real_P'] / d['real_D']
    _CACHE['shiller'] = d
    return d


def _french_table(fname, section=None, scale=100.0):
    """Un tabel lunar (YYYYMM) dintr-un fisier Kenneth French: primul (implicit) sau cel cu titlul `section`;
    randamentele sunt impartite la `scale` (100: in zecimal), numarul de firme se citeste cu scale=1."""
    raw = urllib.request.urlopen(urllib.request.Request(FRENCH + fname, headers={'User-Agent': 'Mozilla/5.0'}),
                                 timeout=120).read()
    text = zipfile.ZipFile(io.BytesIO(raw)).read(zipfile.ZipFile(io.BytesIO(raw)).namelist()[0]).decode('latin-1')
    lines = text.splitlines()
    k0 = 0 if section is None else next(k for k, l in enumerate(lines) if l.strip() == section)
    i = next(k for k, l in enumerate(lines) if k >= k0 and l.startswith(',') and len(l.split(',')) > 1)
    cols = [c.strip() for c in lines[i].split(',')[1:]]
    rows = []
    for l in lines[i + 1:]:
        parts = [x.strip() for x in l.split(',')]
        if not parts or len(parts[0]) != 6 or not parts[0].isdigit():
            break
        rows.append([parts[0]] + [float(x) for x in parts[1:]])
    df = pd.DataFrame(rows, columns=['date'] + cols)
    df['date'] = pd.to_datetime(df['date'], format='%Y%m') + pd.offsets.MonthEnd(0)
    return df.set_index('date').replace([-99.99, -999.0], np.nan) / scale


def industries():
    """49 de industrii (randamente lunare ponderate cu valoarea) si randamentul pietei (Mkt-RF + RF)."""
    if 'ind' in _CACHE:
        return _CACHE['ind']
    ind = _french_table('49_Industry_Portfolios_CSV.zip')
    ff = _french_table('F-F_Research_Data_Factors_CSV.zip')
    mkt = (ff['Mkt-RF'] + ff['RF']).rename('MKT')
    _CACHE['ind'] = (ind, mkt)
    return ind, mkt


def industry_firms():
    """Numarul lunar de firme din fiecare dintre cele 49 de portofolii sectoriale (Kenneth French)."""
    if 'firms' not in _CACHE:
        _CACHE['firms'] = _french_table('49_Industry_Portfolios_CSV.zip', 'Number of Firms in Portfolios', 1.0)
    return _CACHE['firms']

## Explosiveness tests, LPPLS and drawdowns

In [ ]:
def _cums(y):
    """Sume cumulate pentru regresia Delta y_t = a + b y_{t-1} (randurile t = 1..n); y poate fi (n+1,) sau (R, n+1)."""
    y = np.atleast_2d(np.asarray(y, float))
    x, z = y[:, :-1], np.diff(y, axis=1)
    pad = lambda a: np.concatenate([np.zeros((a.shape[0], 1)), np.cumsum(a, axis=1)], axis=1)
    return pad(x), pad(x * x), pad(z), pad(z * z), pad(x * z)


def _adf_end(C, e, s):
    """Statistica t a lui b pentru ferestrele [s, e] (s vector), pe toate replicile (randurile lui C)."""
    Sx, Sxx, Sz, Szz, Sxz = (c[:, e + 1][:, None] - c[:, s] for c in C)
    n = (e + 1 - s).astype(float)
    den = n * Sxx - Sx ** 2
    b = (n * Sxz - Sx * Sz) / den
    a = (Sz - b * Sx) / n
    ssr = np.maximum(Szz - a * Sz - b * Sxz, 1e-300)
    return b / np.sqrt(ssr / (n - 2) * n / den)


def adf_stat(y):
    """ADF la dreapta pe intreaga serie (fara lag-uri)."""
    C = _cums(y)
    n = len(y) - 1
    return float(_adf_end(C, n - 1, np.array([0]))[0, 0])


def min_window(T, r0=None):
    """Fereastra minima PSY: r0 = 0.01 + 1.8 / sqrt(T)."""
    r0 = 0.01 + 1.8 / np.sqrt(T) if r0 is None else r0
    return int(np.floor(r0 * T)), r0


def psy(y, r0=None):
    """ADF, SADF, GSADF si sirurile BSADF (sup pe inceputuri) si ADF recursiv (PWY) pentru y (log-nivel)."""
    y = np.asarray(y, float)
    n = len(y) - 1                                  # numarul de randuri de regresie
    w0, r0 = min_window(n, r0)
    C = _cums(y)
    bsadf = np.full(n, np.nan)
    fwd = np.full(n, np.nan)
    for e in range(w0 - 1, n):
        s = np.arange(0, e - w0 + 2)
        st = _adf_end(C, e, s)[0]
        bsadf[e] = st.max()
        fwd[e] = st[0]
    return dict(adf=float(fwd[-1]), sadf=float(np.nanmax(fwd)), gsadf=float(np.nanmax(bsadf)),
                bsadf=bsadf, fwd=fwd, w0=w0, r0=r0, n=n)


def _null_paths(T, R, rng):
    """Mers aleator cu drift slab (PSY 2015): y_t = T^{-1} + y_{t-1} + e_t, e_t ~ N(0, 1)."""
    e = rng.standard_normal((R, T))
    return np.concatenate([np.zeros((R, 1)), np.cumsum(1.0 / T + e, axis=1)], axis=1)


def _bsadf_paths(Y, w0, chunk=100):
    """Sirurile BSADF si ADF recursiv pentru mai multe traiectorii (randurile lui Y)."""
    R, n = Y.shape[0], Y.shape[1] - 1
    bs = np.full((R, n), np.nan)
    fw = np.full((R, n), np.nan)
    for i in range(0, R, chunk):
        C = _cums(Y[i:i + chunk])
        for e in range(w0 - 1, n):
            st = _adf_end(C, e, np.arange(0, e - w0 + 2))
            bs[i:i + chunk, e] = st.max(axis=1)
            fw[i:i + chunk, e] = st[:, 0]
    return bs, fw


def psy_cv(T, w0, R=1000, seed=42, q=(0.90, 0.95, 0.99)):
    """Valori critice Monte Carlo: ADF, SADF, GSADF (cuantile) si sirurile BSADF / ADF recursiv la 95%."""
    rng = np.random.default_rng(seed)
    bs, fw = _bsadf_paths(_null_paths(T, R, rng), w0)
    out = dict(T=T, w0=w0, R=R,
               adf={f'{int(100 * a)}': float(np.quantile(fw[:, -1], a)) for a in q},
               sadf={f'{int(100 * a)}': float(np.quantile(np.nanmax(fw, axis=1), a)) for a in q},
               gsadf={f'{int(100 * a)}': float(np.quantile(np.nanmax(bs, axis=1), a)) for a in q})
    out['bsadf95'] = np.nanquantile(bs, 0.95, axis=0)
    out['fwd95'] = np.nanquantile(fw, 0.95, axis=0)
    out['null'] = dict(adf=fw[:, -1], sadf=np.nanmax(fw, axis=1), gsadf=np.nanmax(bs, axis=1))
    return out


def wild_cv(y, w0, R=500, seed=42):
    """Wild bootstrap (semne Rademacher pe Delta y, fara drift): sirul BSADF la 95% si GSADF la 95%."""
    rng = np.random.default_rng(seed)
    dy = np.diff(np.asarray(y, float))
    dy = dy - dy.mean()
    W = rng.choice([-1.0, 1.0], size=(R, len(dy)))
    Y = np.concatenate([np.zeros((R, 1)), np.cumsum(W * dy, axis=1)], axis=1)
    bs, _ = _bsadf_paths(Y, w0)
    return dict(bsadf95=np.nanquantile(bs, 0.95, axis=0), gsadf95=float(np.quantile(np.nanmax(bs, axis=1), 0.95)))


def episodes(stat, cv, index, min_len):
    """Episoade in care stat > cv cel putin min_len perioade: (inceput, sfarsit) datat in timp real."""
    above = np.asarray(stat > cv)
    above[np.isnan(stat) | np.isnan(cv)] = False
    out, i, n = [], 0, len(above)
    while i < n:
        if above[i]:
            j = i
            while j + 1 < n and above[j + 1]:
                j += 1
            if j - i + 1 >= min_len:
                out.append((index[i], index[j] if j + 1 < n else None, j - i + 1))
            i = j + 1
        else:
            i += 1
    return out


# =============================================================================
# BULE RATIONALE SIMULATE
# =============================================================================
def blanchard_watson(T=400, r=0.01, pi=0.97, b0=1.0, sd=0.5, seed=7):
    """Bula Blanchard-Watson: supravietuieste cu prob. pi si creste cu (1+r)/pi, altfel revine la zgomot."""
    rng = np.random.default_rng(seed)
    b = np.empty(T)
    b[0] = b0
    for t in range(1, T):
        eps = sd * rng.standard_normal()
        b[t] = (1 + r) / pi * b[t - 1] + eps if rng.random() < pi else eps
    return b


def evans_bubble(T=400, r=0.02, alpha=1.0, delta=0.5, pi=0.85, seed=11):
    """Bula care se prabuseste periodic (Evans 1991): creste cu (1+r) sub alpha, apoi explodeaza sau revine la delta."""
    rng = np.random.default_rng(seed)
    u = np.exp(rng.normal(-0.5 * 0.07 ** 2, 0.07, T))   # u_t > 0, E[u_t] = 1 exact (lognormal)
    B = np.empty(T)
    B[0] = delta
    for t in range(1, T):
        if B[t - 1] <= alpha:
            B[t] = (1 + r) * B[t - 1] * u[t]
        else:
            theta = rng.random() < pi
            B[t] = (delta + (1 + r) / pi * (B[t - 1] - delta / (1 + r)) * theta) * u[t]
    return B


# =============================================================================
# LPPLS (Johansen-Ledoit-Sornette; calibrare Filimonov-Sornette 2013)
# Spatiul de cautare, filtrele si ferestrele: Shu & Zhu (2020), Physica A 557, 124892, sectiunea 2.2,
# ecuatiile (11)-(12), dupa Sornette et al. (2015)
# =============================================================================
LPPLS_SEARCH = dict(m=(0.0, 1.0), w=(1.0, 50.0), tc_frac=(0.0, 1 / 3), damping_min=1.0)            # ec. (11)
LPPLS_FILTER = dict(m=(0.01, 0.99), w=(2.0, 25.0), tc_frac=(0.0, 1 / 5), osc_min=2.5,             # ec. (12)
                    rel_err_max=0.15, lomb_alpha=0.10, ar1_alpha=0.10)
LPPLS_WINDOWS = list(range(750, 45, -5))       # t2 - t1 de la 750 la 50 de observatii, pas 5: 141 de ferestre
LPPLS_STEP = 5                                 # t2 se muta cu 5 observatii


def lppls_design(t, tc, m, w):
    dt = np.maximum(tc - t, 1e-9)
    f = dt ** m
    lg = np.log(dt)
    return np.column_stack([np.ones_like(t), f, f * np.cos(w * lg), f * np.sin(w * lg)])


def lppls_linear(t, y, tc, m, w):
    """Pentru (tc, m, omega) dati: A, B, C1, C2 prin MCO si suma patratelor reziduurilor."""
    X = lppls_design(t, tc, m, w)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ beta
    return beta, float(res @ res)


def _damping(m, w, beta):
    C = np.hypot(beta[..., 2], beta[..., 3])
    return m * np.abs(beta[..., 1]) / (w * np.maximum(C, 1e-300))


def _grid(t, y, tcs, ms, ws):
    TC, M, W = (a.ravel() for a in np.meshgrid(tcs, ms, ws, indexing='ij'))
    dt = np.maximum(TC[:, None] - t[None, :], 1e-9)
    f = dt ** M[:, None]
    lg = np.log(dt)
    X = np.stack([np.ones_like(f), f, f * np.cos(W[:, None] * lg), f * np.sin(W[:, None] * lg)], axis=2)
    XtX = np.einsum('gni,gnj->gij', X, X) + 1e-10 * np.eye(4)
    Xty = np.einsum('gni,n->gi', X, y)
    beta = np.linalg.solve(XtX, Xty[..., None])[..., 0]
    ssr = y @ y - np.einsum('gi,gi->g', beta, Xty)
    return TC, M, W, beta, ssr


def lppls_fit(t, y, search=LPPLS_SEARCH, refine=True, grid=(8, 8, 16)):
    """Calibrare LPPLS pe fereastra (t, y), t in ani: minimul SSR in spatiul de cautare al ec. (11)
    (tc in [t2, t2 + (t2 - t1)/3], m in [0, 1], omega in [1, 50], amortizare >= 1): grila + Nelder-Mead."""
    t = np.asarray(t, float)
    y = np.asarray(y, float)
    t1, t2 = t[0], t[-1]
    D = t2 - t1
    lo = np.array([t2 + search['tc_frac'][0] * D + 1e-6, search['m'][0] + 1e-3, search['w'][0]])
    hi = np.array([t2 + search['tc_frac'][1] * D, search['m'][1] - 1e-3, search['w'][1]])
    TC, M, W, beta, ssr = _grid(t, y, np.linspace(lo[0], hi[0], grid[0]), np.linspace(lo[1], hi[1], grid[1]),
                                np.linspace(lo[2], hi[2], grid[2]))
    ok = _damping(M, W, beta) >= search['damping_min']
    if not ok.any():
        return None
    k = int(np.argmin(np.where(ok, ssr, np.inf)))
    x0 = np.array([TC[k], M[k], W[k]])
    if refine:
        def obj(p):
            q = np.clip(p, lo, hi)
            b, sr = lppls_linear(t, y, *q)
            return sr if _damping(q[1], q[2], b) >= search['damping_min'] else sr + 1e6
        from scipy import optimize
        r = optimize.minimize(obj, x0, method='Nelder-Mead', options=dict(xatol=1e-5, fatol=1e-10, maxiter=400))
        if r.fun < 1e6:
            x0 = np.clip(r.x, lo, hi)
    tc, m, w = x0
    (A, B, C1, C2), s = lppls_linear(t, y, tc, m, w)
    C = np.hypot(C1, C2)
    yhat = lppls_design(t, tc, m, w) @ np.array([A, B, C1, C2])
    return dict(tc=float(tc), m=float(m), w=float(w), A=float(A), B=float(B), C1=float(C1), C2=float(C2), C=float(C),
                ssr=float(s), rmse=float(np.sqrt(s / len(t))), damping=float(m * abs(B) / (w * C)) if C > 0 else np.inf,
                osc=float(w / np.pi * np.log((tc - t1) / (tc - t2))),
                rel_err=float(np.max(np.abs(np.exp(yhat) - np.exp(y)) / np.exp(y))), t1=float(t1), t2=float(t2),
                _t=t, _y=y, _yhat=yhat)


def lomb_pvalue(fit, wmin=2.0, wmax=25.0, nw=200):
    """Testul Lomb: rezidualul fara tendinta r = (tc - t)^(-m) (ln p - A - B (tc - t)^m) ca functie de ln(tc - t);
    probabilitatea ca varful maxim al periodogramei normalizate sa apara intamplator."""
    from scipy.signal import lombscargle
    t, y = fit['_t'], fit['_y']
    dt = np.maximum(fit['tc'] - t, 1e-9)
    r = dt ** (-fit['m']) * (y - fit['A'] - fit['B'] * dt ** fit['m'])
    x = np.log(dt)
    r = r - r.mean()
    ws = np.linspace(wmin, wmax, nw)
    p = lombscargle(x, r, ws) / r.var()                          # periodograma normalizata (Scargle 1982): P_N = P / var(r)
    z = p.max()
    M = min(nw, len(r))
    return float(1 - (1 - np.exp(-z)) ** M)


def ar1_pass(fit, alpha):
    """Reziduul ln(p_hat) - ln(p) este AR(1) (Ornstein-Uhlenbeck): testele Dickey-Fuller si Phillips-Perron resping
    radacina unitara la nivelul alpha."""
    from statsmodels.tsa.stattools import adfuller
    from arch.unitroot import PhillipsPerron
    e = fit['_yhat'] - fit['_y']
    return bool(adfuller(e, maxlag=0, autolag=None, regression='c')[1] < alpha and PhillipsPerron(e, trend='c').pvalue < alpha)


def lppls_conditions(fit, flt=LPPLS_FILTER, search=LPPLS_SEARCH):
    """Conditiile ec. (12) (plus amortizarea din ec. 11); bula pozitiva: B < 0. Conditiile de parametri si eroarea relativa
    se evalueaza fiecare separat; testul Lomb doar daca toate trec, iar testul de radacina unitara al reziduului doar daca
    trece si Lomb (ponderi cumulative pentru ultimele doua)."""
    if fit is None:
        return None
    D = fit['t2'] - fit['t1']
    c = dict(B=fit['B'] < 0, m=flt['m'][0] <= fit['m'] <= flt['m'][1], w=flt['w'][0] <= fit['w'] <= flt['w'][1],
             tc=fit['t2'] + flt['tc_frac'][0] * D <= fit['tc'] <= fit['t2'] + flt['tc_frac'][1] * D,
             osc=fit['osc'] >= flt['osc_min'], damping=fit['damping'] >= search['damping_min'],
             rel_err=fit['rel_err'] <= flt['rel_err_max'])
    if all(c.values()):
        c['lomb'] = lomb_pvalue(fit) <= flt['lomb_alpha']
        c['ar1'] = ar1_pass(fit, flt['ar1_alpha']) if c['lomb'] else False
    else:
        c['lomb'] = c['ar1'] = None
    return c


def lppls_qualified(fit, flt=LPPLS_FILTER):
    c = lppls_conditions(fit, flt)
    return bool(c is not None and all(v is True for v in c.values()))


def lppls_path(fit, t):
    """Traiectoria LPPLS ajustata (log-pret), pentru t < tc."""
    t = np.asarray(t, float)
    return lppls_design(t, fit['tc'], fit['m'], fit['w']) @ np.array([fit['A'], fit['B'], fit['C1'], fit['C2']])


def lppls_confidence(t, y, t2_idx, windows=LPPLS_WINDOWS):
    """Indicatorul de incredere LPPLS (bule pozitive) la t2: ponderea ferestrelor [t2 - L, t2] cu ajustari calificate."""
    q = []
    for L in windows:
        i1 = t2_idx - L
        if i1 < 0:
            continue
        f = lppls_fit(t[i1:t2_idx + 1], y[i1:t2_idx + 1])
        q.append(lppls_qualified(f))
    return float(np.mean(q)) if q else np.nan


# =============================================================================
# DRAWDOWN
# =============================================================================
def drawdown(p):
    """Scaderea fata de maximul anterior: p_t / max_{s<=t} p_s - 1."""
    return p / p.cummax() - 1

## Helper functions

In [ ]:
def _chg(y, s, e):
    """Variatia pretului pe durata episodului (de la inceputul la sfarsitul semnalului): > 0 crestere, < 0 prabusire."""
    e = y.index[-1] if e is None else e
    return float(np.exp(y.loc[e] - y.loc[s]) - 1)


def run_psy(y, R=R_MC, wild=False):
    """PSY complet pentru o serie log-nivel y (pd.Series): statistici, valori critice, episoade datate."""
    res = psy(y.values)
    cv = psy_cv(res['n'], res['w0'], R, SEED)
    L = int(np.ceil(np.log(res['n'])))
    idx = y.index[1:]
    ep = episodes(res['bsadf'], cv['bsadf95'], idx, L)
    ep_pwy = episodes(res['fwd'], cv['fwd95'], idx, L)
    out = dict(res=res, cv=cv, L=L, idx=idx, ep=ep, ep_pwy=ep_pwy)
    if wild:
        wc = wild_cv(y.values, res['w0'], 1000, SEED)
        out['wild'] = wc
        out['ep_wild'] = episodes(res['bsadf'], wc['bsadf95'], idx, L)
    return out


def summarise(o, y):
    """Rezumat JSON-abil al unei analize PSY."""
    r, cv = o['res'], o['cv']
    out = dict(T=r['n'], w0=r['w0'], r0=r['r0'], L=o['L'], adf=r['adf'], sadf=r['sadf'], gsadf=r['gsadf'],
               cv_adf=cv['adf'], cv_sadf=cv['sadf'], cv_gsadf=cv['gsadf'], start=d2s(y.index[0]), end=d2s(y.index[-1]),
               episodes=[dict(start=d2s(s), end=d2s(e), n=int(n), change=_chg(y, s, e)) for s, e, n in o['ep']],
               episodes_pwy=[dict(start=d2s(s), end=d2s(e), n=int(n), change=_chg(y, s, e)) for s, e, n in o['ep_pwy']],
               first_window_end=d2s(o['idx'][r['w0'] - 1]))
    if 'wild' in o:
        out['cv_gsadf_wild'] = o['wild']['gsadf95']
        out['episodes_wild'] = [dict(start=d2s(s), end=d2s(e), n=int(n), change=_chg(y, s, e)) for s, e, n in o['ep_wild']]
    return out


def real_time(o, p, win_years=2):
    """Pentru fiecare episod: primul semnal (timp real), varful pretului in episod si dupa, sfarsitul semnalului."""
    rows = []
    for s, e, n in o['ep']:
        e2 = e if e is not None else p.index[-1]
        seg = p.loc[s:e2 + pd.DateOffset(months=6)]
        pk = seg.idxmax()
        rows.append(dict(start=d2s(s), end=d2s(e), peak=d2s(pk), lead_days=int((pk - s).days),
                         change=float(p.loc[:e2].iloc[-1] / p.loc[:s].iloc[-1] - 1),
                         gain_to_peak=float(p.loc[pk] / p.loc[:s].iloc[-1] - 1),
                         dd_after=float((p.loc[pk:pk + pd.DateOffset(years=win_years)].min()) / p.loc[pk] - 1),
                         signal_end_after_peak_days=None if e is None else int((e - pk).days)))
    return rows


def ms_fit(key, start, freq='W', k=2):
    """Markov-switching: medie si dispersie diferite pe regim, randamente log saptamanale in %."""
    import statsmodels.api as sm
    p = price(key, freq, start)
    r = 100 * np.log(p).diff().dropna()
    np.random.seed(SEED)                              # cautarea aleatoare a punctelor de start: reproductibila
    res = sm.tsa.MarkovRegression(r, k_regimes=k, trend='c', switching_variance=True).fit(search_reps=20, disp=False)
    hi = int(np.argmax([res.params[f'sigma2[{j}]'] for j in range(k)]))
    return p, r, res, hi


def ms_summary(res, hi, r, k=2):
    par = res.params
    se = res.bse
    lo = 1 - hi if k == 2 else None
    P = np.asarray(res.regime_transition)[:, :, 0]         # P[i, j] = P(s_t = i | s_{t-1} = j)
    out = dict(n=int(len(r)), start=d2s(r.index[0]), end=d2s(r.index[-1]), llf=float(res.llf), aic=float(res.aic),
               bic=float(res.bic))
    for j, tag in [(lo, 'calm'), (hi, 'turb')]:
        out[tag] = dict(mu=float(par[f'const[{j}]']), mu_se=float(se[f'const[{j}]']),
                        sd=float(np.sqrt(par[f'sigma2[{j}]'])), stay=float(P[j, j]),
                        dur=float(res.expected_durations[j]), share=float(res.smoothed_marginal_probabilities[j].mean()))
    return out


def ci_evaluation(ci, p, horizon=90, fall=0.20):
    """Semnal (indicator > 0) vs o scadere de cel putin 20% sub pretul curent in urmatoarele 90 de zile;
    doar datele cu fereastra completa de 90 de zile calendaristice in date."""
    rows = []
    for d, v in ci.items():
        if d + pd.Timedelta(days=horizon) > p.index[-1]:
            continue
        fut = p.loc[d:d + pd.Timedelta(days=horizon)]
        rows.append(dict(date=d, ci=v, hit=bool(fut.min() / fut.iloc[0] - 1 <= -fall)))
    r = pd.DataFrame(rows)
    on, off = r[r.ci > 0], r[r.ci == 0]
    return dict(n=int(len(r)), n_on=int(len(on)), p_on=float(on.hit.mean()) if len(on) else None,
                p_off=float(off.hit.mean()), base=float(r.hit.mean()), horizon=horizon, fall=fall)

## Analysis

In [ ]:
def a1_present_value(D1=2.0, r=0.06, g=0.02, P=65.0):
    """Gordon: F = D1 / (r - g); bula B = P - F creste cu r, fundamentalul cu g."""
    F = D1 / (r - g)
    B = P - F
    out = dict(F=F, B=B, share0=B / P)
    for h in (5, 30, 100):
        Fh, Bh = F * (1 + g) ** h, B * (1 + r) ** h
        out[f'F{h}'], out[f'B{h}'], out[f'share{h}'] = Fh, Bh, Bh / (Fh + Bh)
    return out


def a2_blanchard_watson(B0=15.0, r=0.06, pi=0.9, h=5):
    """Bula Blanchard-Watson: supravietuieste cu pi pe an; conditionat de supravietuire creste cu (1+r)/pi."""
    return dict(growth=(1 + r) / pi - 1, surv=pi ** h, life=1 / (1 - pi), EB=B0 * (1 + r) ** h,
                B_surv=B0 * ((1 + r) / pi) ** h, pi=pi, h=h,
                csd=B0 * (1 + r) * np.sqrt((1 - pi) / pi),          # abaterea standard conditionata (fara zgomot)
                log_slope=-(1 - pi))                                # panta E[Delta y | y] in logaritmi


def _df_t(Y):
    """Statistica t a lui b din Delta y_t = a + b y_{t-1} + e_t, pe fiecare rand al lui Y."""
    X, Z = Y[:, :-1], np.diff(Y, axis=1)
    n = Z.shape[1]
    sx, sxx, sz, sxz, szz = X.sum(1), (X * X).sum(1), Z.sum(1), (X * Z).sum(1), (Z * Z).sum(1)
    dd = n * sxx - sx ** 2
    bb = (n * sxz - sx * sz) / dd
    aa = (sz - bb * sx) / n
    return bb / np.sqrt((szz - aa * sz - bb * sxz) / (n - 2) * n / dd)


def a3_df_limit(T=1000, R=20_000):
    """Distributia Dickey-Fuller cu termen liber: simulare pentru T mare; media numaratorului limita = -1/2."""
    rng = np.random.default_rng(SEED)
    tt = np.concatenate([_df_t(np.cumsum(rng.standard_normal((2000, T + 1)), axis=1)) for _ in range(R // 2000)])
    # functionala limita aproximata pe grila de T puncte: numarator 1/2 (W(1)^2 - 1) - W(1) int W
    W = np.cumsum(rng.standard_normal((5000, T)), axis=1) / np.sqrt(T)
    num = 0.5 * (W[:, -1] ** 2 - 1) - W[:, -1] * W.mean(1)
    return dict(T=T, R=R, q05=float(np.quantile(tt, 0.05)), q50=float(np.quantile(tt, 0.50)),
                q95=float(np.quantile(tt, 0.95)), q99=float(np.quantile(tt, 0.99)),
                p_neg=float((tt < 0).mean()), num_mean=float(num.mean()))


def a4_mild(Ts=(100, 400, 1600), alpha=0.8, c=1.0, R=4000):
    """Radacina usor exploziva rho_T = 1 + c / T^alpha: mediana statisticii ADF creste cu T (divergenta la +inf)."""
    rng = np.random.default_rng(SEED)
    out = []
    for T in Ts:
        rho = 1 + c / T ** alpha
        e = rng.standard_normal((R, T))
        Y = np.zeros((R, T + 1))
        for t in range(T):
            Y[:, t + 1] = rho * Y[:, t] + e[:, t]
        tt = _df_t(Y)
        out.append(dict(T=T, rho=rho, rhoT=float(rho ** T), med=float(np.median(tt)), rej=float((tt > 1.28).mean())))
    return out


def a4_windows(T=440):
    """(pastrat pentru notebook) ferestrele PSY pentru T = 440."""
    w0, r0 = min_window(T)
    n_sadf = T - w0 + 1
    return dict(T=T, r0=r0, w0=w0, n_sadf=n_sadf, n_gsadf=n_sadf * (n_sadf + 1) // 2, logT=float(np.log(T)),
                L=int(np.ceil(np.log(T))))


def _ms_step(P, mu, sd, prev_turb, r):
    """Un pas al filtrului Hamilton cu doua regimuri (0 = calm, 1 = turbulent)."""
    prev = np.array([1 - prev_turb, prev_turb])
    pred = P.T @ prev                          # P[i, j] = P(s_t = j | s_{t-1} = i)
    lik = stats.norm.pdf(r, mu, sd)
    post = pred * lik / (pred * lik).sum()
    return pred, lik, post


def a5_markov(P=((0.98, 0.02), (0.10, 0.90)), mu=(0.3, -0.5), sd=(2.0, 5.0), prev_turb=0.2, r=-6.0):
    P, mu, sd = np.array(P), np.array(mu), np.array(sd)
    dur = 1 / (1 - np.diag(P))
    pi_turb = P[0, 1] / (P[0, 1] + P[1, 0])
    pi = np.array([1 - pi_turb, pi_turb])
    m = pi @ mu
    v = pi @ (sd ** 2) + pi @ (mu - m) ** 2
    pred, lik, post = _ms_step(P, mu, sd, prev_turb, r)
    return dict(dur_calm=dur[0], dur_turb=dur[1], pi_calm=pi[0], pi_turb=pi[1], mean=m, var=v, sd=np.sqrt(v),
                pred_turb=pred[1], lik_calm=lik[0], lik_turb=lik[1], post_turb=post[1], r=r, prev=prev_turb)


def a6_markov(P=((0.95, 0.05), (0.20, 0.80)), mu=(0.4, -1.0), sd=(2.0, 6.0), prev_turb=0.1, r=1.0):
    out = a5_markov(P, mu, sd, prev_turb, r)
    P = np.array(P)
    two = np.array([1 - out['post_turb'], out['post_turb']]) @ np.linalg.matrix_power(P, 2)
    out['two_step_turb'] = two[1]
    return out


def _lppls_diag(t1, t2, tc, m, w, B, C1, C2):
    """Conditiile din Shu & Zhu (2020), ec. (11)-(12), care se pot verifica din parametri (fara date)."""
    C = np.hypot(C1, C2)
    D = t2 - t1
    O = w / np.pi * np.log((tc - t1) / (tc - t2))          # numarul de semiperioade (Shu & Zhu 2020, ec. 12)
    damp = m * abs(B) / (w * C)
    exact = m * abs(B) / (C * np.hypot(m, w))              # rata de hazard >= 0 pentru orice faza: m|B| >= |C| sqrt(m^2 + w^2)
    slope = -B * m * (tc - t2) ** (m - 1)          # d/dt [B (tc - t)^m] la t2, fara oscilatii
    return dict(C=C, D=D, O=O, damping=damp, exact=exact, dtc=tc - t2, dtc_days=(tc - t2) * 365.25, tc_lim=D / 5,
                slope=slope, m=m, w=w, B=B, ok_B=B < 0, ok_m=0.01 <= m <= 0.99, ok_w=2 <= w <= 25,
                ok_tc=0 <= tc - t2 <= D / 5, ok_O=O >= 2.5, ok_D=damp >= 1, ok_exact=exact >= 1)


def a7_lppls():
    return _lppls_diag(2025.0, 2026.0, 2026.08, 0.5, 8.0, -1.2, 0.06, -0.02)


def a8_lppls():
    return _lppls_diag(2025.0, 2026.0, 2026.45, 0.95, 3.5, -0.8, 0.25, 0.0)


def b1_pd_windows(r0s=(0.03, None, 0.08)):
    """GSADF pe raportul P/D Shiller pentru trei ferestre minime (regula PSY in mijloc)."""
    sh = shiller()
    y = np.log(sh['PD'])
    out = []
    for r0 in r0s:
        res = psy(y.values, r0)
        cv = psy_cv(res['n'], res['w0'], 1000, SEED)
        L = int(np.ceil(np.log(res['n'])))
        ep = episodes(res['bsadf'], cv['bsadf95'], y.index[1:], L)
        out.append(dict(r0=res['r0'], w0=res['w0'], gsadf=res['gsadf'], cv95=cv['gsadf']['95'], cv99=cv['gsadf']['99'],
                        episodes=[dict(start=d2s(s), end=d2s(e), n=int(n), change=_chg(y, s, e)) for s, e, n in ep]))
    return out


def b2_sadf_vs_gsadf():
    """Nasdaq 100 si S&P 500 lunar: datarea PWY (inceput fix) vs PSY (inceput mobil)."""
    out = {}
    for k in ['ndx', 'sp500']:
        y = np.log(price(k, 'M'))
        o = run_psy(y)
        out[k] = summarise(o, y)
    return out


def b3_btc_wild():
    """Bitcoin saptamanal: valori critice Monte Carlo (mers aleator gaussian) vs wild bootstrap."""
    y = np.log(price('btc', 'W'))
    o = run_psy(y, wild=True)
    idx = o['idx']
    dy = np.diff(y.values)
    vol = pd.Series(dy, index=idx).rolling(26).std() * np.sqrt(52)
    fig, axes = plt.subplots(2, 1, figsize=(7.4, 4.0), sharex=True, gridspec_kw=dict(height_ratios=[1, 1.2]))
    axes[0].plot(vol.index, 100 * vol.values, color=Teal, lw=0.9, label='Annualised volatility, 26-week window (%)')
    axes[0].set_ylabel('%')
    axes[1].plot(idx, o['res']['bsadf'], color=IDAred, lw=0.9, label='BSADF statistic')
    axes[1].plot(idx, o['cv']['bsadf95'], color=Forest, lw=1.0, ls='--', label='95% critical value, Monte Carlo')
    axes[1].plot(idx, o['wild']['bsadf95'], color=Purple, lw=1.0, ls='-.', label='95% critical value, wild bootstrap')
    axes[1].set_ylabel('Statistic')
    axes[0].set_title('Bitcoin, weekly 2014-2026: volatility changes the critical values', fontsize=9, loc='left')
    fig.tight_layout()
    fig_legend(fig, axes, ncol=2, y=0.0)
    save_fig('ch17_sem_btc_wild')
    s = summarise(o, y)
    s['vol_first'] = float(vol.dropna().iloc[:52].mean())
    s['vol_last'] = float(vol.dropna().iloc[-52:].mean())
    s['wild_bs_max'] = float(np.nanmax(o['wild']['bsadf95']))
    s['mc_bs_max'] = float(np.nanmax(o['cv']['bsadf95']))
    return s


def b4_bvb():
    """BET lunar 1997-2026 si BET-FI saptamanal 2012-2026: GSADF, episoade (Monte Carlo si wild bootstrap)."""
    out = {}
    fig, axes = plt.subplots(2, 1, figsize=(7.4, 4.2))
    for ax, (k, f, title) in zip(axes, [('bet', 'M', 'BET, monthly 1997-2026'), ('betfi', 'W', 'BET-FI, weekly 2012-2026')]):
        y = np.log(price(k, f))
        o = run_psy(y, wild=True)
        out[k] = summarise(o, y)
        out[k]['real_time'] = real_time(o, price(k, 'D'))
        ax.plot(o['idx'], o['res']['bsadf'], color=IDAred, lw=0.9, label='BSADF statistic')
        ax.plot(o['idx'], o['cv']['bsadf95'], color=Forest, lw=1.0, ls='--', label='95% critical value, Monte Carlo')
        ax.plot(o['idx'], o['wild']['bsadf95'], color=Purple, lw=1.0, ls='-.', label='95% critical value, wild bootstrap')
        ax.set_title(title, fontsize=8.5, loc='left')
        ax.set_ylabel('Statistic')
    fig.tight_layout()
    fig_legend(fig, axes, ncol=3, y=0.0)
    save_fig('ch17_sem_bvb')
    return out


def b5_tc_windows(t2='2017-11-15', starts=('2016-06-01', '2017-08-01')):
    """LPPLS pe Bitcoin cu t2 fix si inceputul ferestrei t1 mobil (saptamanal): distributia lui tc."""
    p = price('btc', 'D')
    rows = []
    for t1 in pd.date_range(starts[0], starts[1], freq='7D'):
        w = p.loc[t1:t2]
        f = lppls_fit(yrs(w.index), np.log(w.values))
        rows.append(dict(t1=w.index[0], tc=todate(f['tc']), m=f['m'], w=f['w'], q=lppls_qualified(f),
                         at_bound=bool(f['m'] <= 0.002 or f['m'] >= 0.998 or f['w'] <= 1.01 or f['w'] >= 49.9)))
    d = pd.DataFrame(rows)
    pk = p.loc['2017'].idxmax()
    err = (d['tc'] - pk).dt.days
    fig, ax = plt.subplots(figsize=(7.2, 3.0))
    ax.plot(d['t1'], d['tc'], 'o', color=IDAred, ms=3.5, label='Estimated $t_c$ (window start $t_1$ on the x-axis)')
    ax.plot(d.loc[d['q'], 't1'], d.loc[d['q'], 'tc'], 'o', mfc='none', mec=MainBlue, ms=7, label='Qualified fit')
    ax.axhline(pk, color=Forest, ls='--', lw=1.0, label=f'Actual peak {pk.date()}')
    ax.axhline(pd.Timestamp(t2), color=Gray, ls=':', lw=0.8)
    ax.set_xlabel('Start of the estimation window $t_1$')
    ax.set_ylabel('Estimated $t_c$')
    ax.set_title(f'Bitcoin, estimation window ends on {t2}: $t_c$ depends on where the window starts', fontsize=9, loc='left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.yaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    legend_outside_bottom(ax, ncol=2, y=-0.22)
    save_fig('ch17_sem_tc_windows')
    return dict(n=int(len(d)), t2=t2, peak=d2s(pk), tc_min=d2s(d['tc'].min()), tc_max=d2s(d['tc'].max()),
                tc_med=d2s(d['tc'].sort_values().iloc[len(d) // 2]), iqr_days=float(np.subtract(*np.percentile(err, [75, 25]))),
                share_q=float(d['q'].mean()), share_bound=float(d['at_bound'].mean()),
                share_30=float((err.abs() <= 30).mean()), med_err=float(err.median()))


def b6_ci_ndx():
    """Indicatorul de incredere LPPLS pentru Nasdaq 100, 1997-2002 (din fisierul publicat de capitol) si evaluarea."""
    path = os.path.join(HERE, 'ch17_lppls_ci_ndx.csv')
    ci = pd.read_csv(path if os.path.exists(path) else QL_RAW + 'ch17_lppls_ci_ndx.csv', index_col=0, parse_dates=True)['ci']
    p = price('ndx', 'D', '1994-01-01')
    ev = ci_evaluation(ci, p)
    w = ci.loc['1997':'2002']
    fig, axes = plt.subplots(2, 1, figsize=(7.4, 3.8), sharex=True, gridspec_kw=dict(height_ratios=[1.3, 1]))
    pp = p.loc['1997':'2002']
    axes[0].plot(pp.index, pp.values, color=MainBlue, lw=0.8, label='Nasdaq 100 (log scale)')
    axes[0].set_yscale('log')
    axes[1].bar(w.index, w.values, width=12, color=IDAred, label='LPPLS confidence indicator')
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel('Share')
    axes[0].set_title('Nasdaq 100, 1997-2002: real-time LPPLS confidence indicator', fontsize=9, loc='left')
    fig.tight_layout()
    fig_legend(fig, axes, ncol=2, y=0.0)
    save_fig('ch17_sem_ci_ndx')
    ev['on_1999_2000'] = float((ci.loc['1999-01-01':'2000-03-27'] > 0).mean())
    ev['first_on'] = d2s(ci[ci > 0].index[0]) if (ci > 0).any() else None
    ev['max'] = float(ci.max())
    ev['max_date'] = d2s(ci.idxmax())
    ev['on_dates'] = [d2s(d) for d in ci[ci > 0].index]
    return ev


def b7_ms_sp():
    """Markov-switching cu doua regimuri pe S&P 500 saptamanal 1990-2026."""
    p, r, res, hi = ms_fit('sp500', '1990-01-01')
    out = ms_summary(res, hi, r)
    fp = res.filtered_marginal_probabilities[hi]
    sp = res.smoothed_marginal_probabilities[hi]
    fig, axes = plt.subplots(2, 1, figsize=(7.4, 3.8), sharex=True, gridspec_kw=dict(height_ratios=[1.2, 1]))
    axes[0].plot(p.index, p.values, color=MainBlue, lw=0.8, label='S&P 500 (log scale)')
    axes[0].set_yscale('log')
    axes[1].plot(fp.index, fp.values, color=IDAred, lw=0.7, label='Filtered probability (full-sample parameters)')
    axes[1].plot(sp.index, sp.values, color=Forest, lw=1.0, label='Smoothed probability (full sample)')
    axes[1].set_ylabel('P(turbulent regime)')
    axes[0].set_title('S&P 500, weekly log returns 1990-2026: two-regime Markov-switching model', fontsize=9, loc='left')
    fig.tight_layout()
    fig_legend(fig, axes, ncol=3, y=0.0)
    save_fig('ch17_sem_ms_sp')
    for tag, a, b in [('dotcom', '2000-01-01', '2000-12-31'), ('gfc', '2008-09-01', '2008-12-31'), ('covid', '2020-02-15', '2020-04-30')]:
        x = fp.loc[a:b]
        out[f'first_{tag}'] = d2s(x[x > 0.5].index[0]) if (x > 0.5).any() else None
    ssp = sp > 0.5
    out['share_turb_smoothed'] = float(ssp.mean())
    out['agree'] = float(((fp > 0.5) == ssp).mean())
    # un regim: aceeasi medie si dispersie
    ll1 = float(stats.norm.logpdf(r, r.mean(), r.std(ddof=0)).sum())
    out['llf1'] = ll1
    out['lr'] = 2 * (out['llf'] - ll1)
    return out


def b8_ms_btc():
    """Bitcoin saptamanal: doua vs trei regimuri (AIC, BIC), media si volatilitatea fiecarui regim."""
    import statsmodels.api as sm
    p = price('btc', 'W')
    r = 100 * np.log(p).diff().dropna()
    out = {}
    for k in (2, 3):
        np.random.seed(SEED + k)                      # cautarea aleatoare a punctelor de start: reproductibila
        res = sm.tsa.MarkovRegression(r, k_regimes=k, trend='c', switching_variance=True).fit(search_reps=30, maxiter=1000, disp=False)
        order = np.argsort([res.params[f'sigma2[{j}]'] for j in range(k)])
        out[f'k{k}'] = dict(llf=float(res.llf), aic=float(res.aic), bic=float(res.bic),
                            mu=[float(res.params[f'const[{j}]']) for j in order],
                            sd=[float(np.sqrt(res.params[f'sigma2[{j}]'])) for j in order],
                            dur=[float(res.expected_durations[j]) for j in order],
                            share=[float(res.smoothed_marginal_probabilities[j].mean()) for j in order])
        if k == 3:
            sp = res.smoothed_marginal_probabilities
            fig, axes = plt.subplots(2, 1, figsize=(7.4, 3.8), sharex=True, gridspec_kw=dict(height_ratios=[1.2, 1]))
            axes[0].plot(p.index, p.values, color=MainBlue, lw=0.8, label='Bitcoin (log scale)')
            axes[0].set_yscale('log')
            for j, c, lab in zip(order, [Forest, Amber, IDAred], ['calm', 'intermediate', 'turbulent']):
                axes[1].plot(sp.index, sp[j].values, color=c, lw=0.8, label=f'Smoothed probability, {lab} regime')
            axes[1].set_ylabel('Probability')
            axes[0].set_title('Bitcoin, weekly 2014-2026: three-regime Markov-switching model', fontsize=9, loc='left')
            fig.tight_layout()
            fig_legend(fig, axes, ncol=2, y=0.0)
            save_fig('ch17_sem_ms_btc3')
    out['n'] = int(len(r))
    return out


def c1_ai():
    """NVIDIA si Nasdaq 100 (lunar si saptamanal) vs Cisco in episodul dot-com: BSADF, cresteri, LPPLS azi."""
    out = {}
    fig, axes = plt.subplots(3, 1, figsize=(7.4, 5.0))
    for ax, (k, a, b) in zip(axes, [('csco', '1990-01-01', '2003-12-31'), ('nvda', '1999-01-01', '2026-09-18'),
                                    ('ndx', '1990-01-01', '2026-09-18')]):
        y = np.log(price(k, 'M'))
        o = run_psy(y, wild=True)
        s = summarise(o, y)
        s['real_time'] = real_time(o, price(k, 'D'))
        out[k] = s
        m = (o['idx'] >= pd.Timestamp(a)) & (o['idx'] <= pd.Timestamp(b))
        ax.plot(o['idx'][m], o['res']['bsadf'][m], color=IDAred, lw=0.9, label='BSADF statistic')
        ax.plot(o['idx'][m], o['cv']['bsadf95'][m], color=Forest, lw=1.0, ls='--', label='95% critical value, Monte Carlo')
        ax.plot(o['idx'][m], o['wild']['bsadf95'][m], color=Purple, lw=1.0, ls='-.', label='95% critical value, wild bootstrap')
        ax.set_title(f'{LABELS[k]}, monthly', fontsize=8.5, loc='left')
    fig.tight_layout()
    fig_legend(fig, axes, ncol=3, y=0.0)
    save_fig('ch17_sem_ai')
    # cresteri pe 2 ani (Greenwood-Shleifer-You), brut si peste Nasdaq 100
    pn = price('nvda', 'M')
    pnd, pcd, pqd = price('nvda', 'D'), price('csco', 'D'), price('ndx', 'D')
    # din preturi zilnice: ultima inchidere pana la data d vs ultima inchidere pana la aceeasi data cu doi ani inainte
    run = lambda p, d: float(p.loc[:d].iloc[-1] / p.loc[:pd.Timestamp(d) - pd.DateOffset(years=2)].iloc[-1] - 1)
    out['run_nvda_now'] = run(pnd, pnd.index[-1])
    out['run_ndx_now'] = run(pqd, pqd.index[-1])
    out['run_csco_peak'] = run(pcd, '2000-03-31')
    out['run_ndx_2000'] = run(pqd, '2000-03-31')
    out['run_nvda_max'] = float((pn / pn.shift(24) - 1).max())
    out['run_nvda_max_date'] = d2s((pn / pn.shift(24) - 1).idxmax())
    pdd = price('csco', 'D')
    pk = pdd.loc['1999':'2001'].idxmax()
    out['csco_peak'] = d2s(pk)
    out['csco_dd'] = float(pdd.loc[pk:].min() / pdd.loc[pk] - 1)
    out['csco_trough'] = d2s(pdd.loc[pk:].idxmin())
    after = pdd.loc[pdd.loc[pk:].idxmin():]                  # dupa minim: prima zi inapoi la nivelul varfului
    out['csco_recover'] = d2s(after[after >= pdd.loc[pk]].index[0]) if (after >= pdd.loc[pk]).any() else None
    nd = price('nvda', 'D')
    out['nvda_ath'] = d2s(nd.idxmax())
    out['nvda_dd_now'] = float(nd.iloc[-1] / nd.max() - 1)
    out['nvda_maxdd_2y'] = float(drawdown(nd.loc['2024-09-18':]).min())
    # LPPLS azi: indicatorul de incredere pe ultimele 26 de saptamani (NVIDIA si Nasdaq 100)
    for k in ['nvda', 'ndx']:
        p = price(k, 'D', '2022-01-01')
        t, yy = yrs(p.index), np.log(p.values)
        idx = np.arange(len(p) - 1, len(p) - 1 - 26 * 5, -10)
        ci = [lppls_confidence(t, yy, i, LPPLS_WINDOWS) for i in idx]
        out[f'ci_{k}_mean'] = float(np.mean(ci))
        out[f'ci_{k}_max'] = float(np.max(ci))
        out[f'ci_{k}_last'] = float(ci[0])
    return out


def fig_sem_data():
    """Datele seminarului: raportul P/D Shiller, Nasdaq 100 lunar, S&P 500 saptamanal, Bitcoin saptamanal."""
    sh = shiller()
    series = [(sh['PD'], 'S&P Composite real price-dividend ratio, monthly', MainBlue),
              (price('ndx', 'M'), 'Nasdaq 100, month-end close', IDAred),
              (price('sp500', 'W', '1990-01-01'), 'S&P 500, week-end close', Forest),
              (price('btc', 'W'), 'Bitcoin (USD), week-end close', Purple)]
    fig, axes = plt.subplots(2, 2, figsize=(7.6, 4.2))
    out = {}
    for ax, (x, lab, col) in zip(axes.ravel(), series):
        x = x.dropna()
        ax.plot(x.index, x.values, color=col, lw=0.8, label=lab)
        ax.set_yscale('log')
        ax.set_title(f'{x.index[0]:%b %Y} - {x.index[-1]:%b %Y}, {len(x)} observations', fontsize=8, loc='left')
        out[lab] = dict(n=int(len(x)), start=d2s(x.index[0]), end=d2s(x.index[-1]))
    fig.tight_layout()
    fig_legend(fig, axes.ravel(), ncol=2, y=0.0)
    save_fig('ch17_sem_data')
    return out


def fig_sem_a1(D1=2.0, r=0.06, g=0.02, P=65.0):
    """A1: valoarea fundamentala F_h, bula asteptata E[B_h] si ponderea bulei in pretul asteptat."""
    a = a1_present_value(D1, r, g, P)
    h = np.arange(0, 101)
    F, B = a['F'] * (1 + g) ** h, a['B'] * (1 + r) ** h
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.9))
    axes[0].plot(h, F, color=MainBlue, label='Fundamental value $F_h$')
    axes[0].plot(h, B, color=IDAred, label='Expected bubble $E[B_h]$')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('Years ahead $h$')
    axes[0].set_ylabel('Value (log scale)')
    axes[1].plot(h, 100 * B / (F + B), color=Purple, label='Bubble share of the expected price (%)')
    for k in (5, 30, 100):
        axes[1].plot(k, 100 * a[f'share{k}'], 'o', color=Purple, ms=4)
        axes[1].annotate(f'{100 * a[f"share{k}"]:.1f}%', (k, 100 * a[f'share{k}']), textcoords='offset points',
                         xytext=(-10, 6), fontsize=7.5, color='black', ha='right' if k == 100 else 'left')
    axes[1].set_ylim(0, 100)
    axes[1].set_xlabel('Years ahead $h$')
    axes[1].set_ylabel('%')
    fig.tight_layout()
    fig_legend(fig, axes, ncol=3, y=0.0)
    save_fig('ch17_sem_a1')
    return {k: a[k] for k in ('F', 'B', 'share5', 'share30', 'share100')}


def fig_sem_a2(B0=15.0, r=0.06, pi=0.9, delta=1.0, T=300, seed=SEED):
    """A2(d): bula Evans (reluare pozitiva delta), fara zgomot; in logaritmi prabusirile arata ca revenire la medie."""
    rng = np.random.default_rng(seed)
    B = np.empty(T + 1)
    B[0] = B0
    for t in range(T):
        B[t + 1] = delta + (1 + r) / pi * (B[t] - delta / (1 + r)) if rng.random() < pi else delta
    y = np.log(B)
    dy, yl = np.diff(y), y[:-1]
    X = np.column_stack([np.ones_like(yl), yl])
    coef = np.linalg.lstsq(X, dy, rcond=None)[0]
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.9))
    axes[0].plot(np.arange(T + 1), B, color=IDAred, lw=0.8, label='Simulated bubble $B_t$ (log scale)')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('Year $t$')
    axes[1].scatter(yl, dy, s=6, color=MainBlue, label=r'$(y_t, \Delta y_{t+1})$, $y = \ln B$')
    xs = np.linspace(yl.min(), yl.max(), 50)
    axes[1].plot(xs, coef[0] + coef[1] * xs, color=Orange, lw=1.2, label=f'OLS fit on all years, slope {coef[1]:.3f}')
    c = np.log(delta)
    axes[1].plot(xs, pi * np.log((1 + r) / pi) + (1 - pi) * (c - xs), color=Forest, lw=1.2, ls='--',
                 label=f'Approximation for $B \\gg \\delta$, slope $-(1-\\pi)$ = {-(1 - pi):.2f}')
    axes[1].axhline(0, color=Gray, lw=0.5)
    axes[1].set_xlabel('$y_t = \\ln B_t$')
    axes[1].set_ylabel('$\\Delta y_{t+1}$')
    fig.tight_layout()
    fig_legend(fig, axes, ncol=3, y=0.0)
    save_fig('ch17_sem_a2')
    return dict(delta=delta, T=T, slope=float(coef[1]), theory=-(1 - pi), collapses=int((np.diff(B) < 0).sum()))


def fig_sem_a3(T=1000, R=20_000):
    """A3: distributia simulata a statisticii t (Dickey-Fuller cu termen liber) si cuantilele ei."""
    rng = np.random.default_rng(SEED)
    tt = np.concatenate([_df_t(np.cumsum(rng.standard_normal((2000, T + 1)), axis=1)) for _ in range(R // 2000)])
    q05, q95 = np.quantile(tt, [0.05, 0.95])
    fig, ax = plt.subplots(figsize=(7.0, 2.8))
    ax.hist(tt, bins=80, density=True, color=MainBlue, alpha=0.8, label=f'Simulated $t_b$, {R:,} random walks, T = {T}')
    ax.axvline(q05, color=Forest, ls='--', lw=1.0, label=f'5% quantile {q05:.2f}')
    ax.axvline(q95, color=IDAred, ls='--', lw=1.2, label=f'95% quantile {q95:.2f} (right-tail critical value)')
    ax.axvline(1.645, color=Orange, ls='-.', lw=1.2, label='1.645, the Normal 95% quantile')
    ax.set_xlabel('$t_b$')
    ax.set_ylabel('Density')
    legend_outside_bottom(ax, ncol=2, y=-0.25)
    save_fig('ch17_sem_a3')
    return dict(q05=float(q05), q95=float(q95), share_neg=float((tt < 0).mean()))


def fig_sem_a4(Ts=(100, 400, 1600), alpha=0.8, c=1.0, R=4000):
    """A4: distributia statisticii t sub o radacina usor exploziva: se muta spre dreapta cand T creste."""
    rng = np.random.default_rng(SEED)
    fig, ax = plt.subplots(figsize=(7.0, 2.8))
    out = []
    for T, col in zip(Ts, [MainBlue, Orange, IDAred]):
        rho = 1 + c / T ** alpha
        e = rng.standard_normal((R, T))
        Y = np.zeros((R, T + 1))
        for t in range(T):
            Y[:, t + 1] = rho * Y[:, t] + e[:, t]
        tt = _df_t(Y)
        ax.hist(np.clip(tt, -4, 40), bins=np.linspace(-4, 40, 89), density=True, histtype='step', lw=1.3, color=col,
                label=f'T = {T}: median {np.median(tt):.1f}')
        out.append(dict(T=T, med=float(np.median(tt))))
    ax.axvline(-0.09, color=Forest, ls='--', lw=1.0, label='Unit-root 95% critical value (A3)')
    ax.set_xlabel('Right-tailed ADF t-statistic (values above 40 shown at 40)')
    ax.set_ylabel('Density')
    legend_outside_bottom(ax, ncol=2, y=-0.25)
    save_fig('ch17_sem_a4')
    return out


def _fig_filter(name, P, mu, sd, prev_turb, r):
    a = a5_markov(P, mu, sd, prev_turb, r)
    x = np.linspace(min(mu) - 4 * max(sd), max(mu) + 4 * max(sd), 400)
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.8), gridspec_kw=dict(width_ratios=[1.6, 1]))
    axes[0].plot(x, stats.norm.pdf(x, mu[0], sd[0]), color=Forest, label=f'Calm density, N({mu[0]}, {sd[0]}$^2$)')
    axes[0].plot(x, stats.norm.pdf(x, mu[1], sd[1]), color=IDAred, label=f'Turbulent density, N({mu[1]}, {sd[1]}$^2$)')
    axes[0].axvline(r, color=MainBlue, ls='--', lw=1.1, label=f'Observed return {r:+.0f}%')
    axes[0].set_xlabel('Weekly return (%)')
    vals = [prev_turb, a['pred_turb'], a['post_turb']]
    bars = axes[1].bar(['Last week', 'Predicted', 'Filtered'], vals, color=[Amber, Orange, Purple],
                       label='P(turbulent regime)')
    for b_, v in zip(bars, vals):
        axes[1].text(b_.get_x() + b_.get_width() / 2, v + 0.02, f'{v:.3f}', ha='center', fontsize=8, color='black')
    axes[1].set_ylim(0, 1.05)
    fig.tight_layout()
    fig_legend(fig, axes, ncol=2, y=0.0)
    save_fig(name)
    return dict(pred=a['pred_turb'], post=a['post_turb'])


def fig_sem_a5():
    return _fig_filter('ch17_sem_a5', ((0.98, 0.02), (0.10, 0.90)), (0.3, -0.5), (2.0, 5.0), 0.2, -6.0)


def fig_sem_a6():
    return _fig_filter('ch17_sem_a6', ((0.95, 0.05), (0.20, 0.80)), (0.4, -1.0), (2.0, 6.0), 0.1, 1.0)


def _lppls_trend(t, tc, m, w, B, C1, C2, osc=True):
    tau = tc - t
    f = tau ** m
    return B * f + (C1 * f * np.cos(w * np.log(tau)) + C2 * f * np.sin(w * np.log(tau)) if osc else 0)


def fig_sem_a7(t1=2025.0, t2=2026.0, tc=2026.08, m=0.5, w=8.0, B=-1.2, C1=0.06, C2=-0.02):
    """A7: tendinta LPPLS implicata de parametri (A = 0): legea putere si oscilatiile log-periodice."""
    t = np.linspace(t1, tc - 1e-4, 2000)
    fig, ax = plt.subplots(figsize=(7.0, 2.8))
    ax.plot(t, _lppls_trend(t, tc, m, w, B, C1, C2, False), color=MainBlue, lw=1.1, label='Power law $B(t_c - t)^m$')
    ax.plot(t, _lppls_trend(t, tc, m, w, B, C1, C2, True), color=IDAred, lw=1.0, label='With log-periodic oscillations')
    ax.axvspan(t1, t2, color=Amber, alpha=0.15, lw=0, label='Estimation window $[t_1, t_2]$')
    ax.axvline(tc, color=Forest, ls='--', lw=1.0, label=f'Critical time $t_c$ = {tc}')
    ax.set_xlabel('Time (years)')
    ax.set_ylabel('$\\ln p - A$')
    legend_outside_bottom(ax, ncol=2, y=-0.25)
    save_fig('ch17_sem_a7')
    d = _lppls_diag(t1, t2, tc, m, w, B, C1, C2)
    return dict(O=d['O'], damping=d['damping'])


def fig_sem_a8():
    """A8: la ce faze este deriva (rata de hazard) negativa? Parametrii A7 vs A8, deriva normalizata cu tau^(1-m)."""
    fig, ax = plt.subplots(figsize=(7.0, 2.8))
    out = {}
    for tag, (t1, t2, tc, m, w, B, C1, C2), col in [('A7', (2025.0, 2026.0, 2026.08, 0.5, 8.0, -1.2, 0.06, -0.02), MainBlue),
                                                   ('A8', (2025.0, 2026.0, 2026.45, 0.95, 3.5, -0.8, 0.25, 0.0), IDAred)]:
        t = np.linspace(t1, t2, 2000)
        tau = tc - t
        C, phi = np.hypot(C1, C2), np.arctan2(C2, C1)
        th = w * np.log(tau) - phi
        drift = -B * m - C * (m * np.cos(th) - w * np.sin(th))       # mu(t) / tau^(m-1)
        ax.plot(t, drift, color=col, lw=1.1, label=f'{tag} parameters: $\\mu(t)\\,\\tau^{{1-m}}$')
        out[tag] = dict(min=float(drift.min()), share_neg=float((drift < 0).mean()))
    ax.axhline(0, color=Gray, lw=0.7)
    ax.set_xlabel('Time (years), estimation window')
    ax.set_ylabel('Normalised drift')
    legend_outside_bottom(ax, ncol=2, y=-0.25)
    save_fig('ch17_sem_a8')
    return out


def fig_sem_b1(r0s=(0.03, None, 0.08)):
    """B1: BSADF pe raportul P/D pentru trei ferestre minime, cu valorile critice si episoadele datate."""
    sh = shiller()
    y = np.log(sh['PD'])
    fig, axes = plt.subplots(3, 1, figsize=(7.4, 5.0), sharex=True)
    out = []
    for ax, r0 in zip(axes, r0s):
        res = psy(y.values, r0)
        cv = psy_cv(res['n'], res['w0'], 1000, SEED)
        L = int(np.ceil(np.log(res['n'])))
        idx = y.index[1:]
        ep = episodes(res['bsadf'], cv['bsadf95'], idx, L)
        ax.plot(idx, res['bsadf'], color=IDAred, lw=0.7, label='BSADF statistic')
        ax.plot(idx, cv['bsadf95'], color=Forest, lw=1.0, ls='--', label='95% critical value (Monte Carlo)')
        for s0, s1, n in ep:
            up = _chg(y, s0, s1) is None or _chg(y, s0, s1) >= 0
            ax.axvspan(s0, s1 if s1 is not None else idx[-1], color=Amber if up else Teal, alpha=0.35, lw=0,
                       label='Episode, ratio rising' if up else 'Episode, ratio falling')
        ax.set_title(f'$r_0$ = {res["r0"]:.3f}, minimum window {res["w0"]} months: GSADF {res["gsadf"]:.2f}, '
                     f'95% critical value {cv["gsadf"]["95"]:.2f}', fontsize=8.5, loc='left')
        ax.set_ylim(-3, 5)
        out.append(dict(r0=res['r0'], w0=res['w0'], gsadf=res['gsadf'], cv95=cv['gsadf']['95'], n_ep=len(ep)))
    fig.tight_layout()
    fig_legend(fig, axes, ncol=2, y=0.0)
    save_fig('ch17_sem_b1')
    return out


def fig_sem_b2():
    """B2: S&P 500 lunar, ADF recursiv (PWY, inceput fix) vs BSADF (PSY), cu valorile critice."""
    y = np.log(price('sp500', 'M'))
    o = run_psy(y)
    idx = o['idx']
    fig, axes = plt.subplots(2, 1, figsize=(7.4, 4.0), sharex=True, gridspec_kw=dict(height_ratios=[1, 1.3]))
    axes[0].plot(y.index, np.exp(y.values), color=MainBlue, lw=0.8, label='S&P 500 (log scale)')
    axes[0].set_yscale('log')
    for s0, s1, n in o['ep']:
        axes[0].axvspan(s0, s1 if s1 is not None else idx[-1], color=Amber, alpha=0.3, lw=0, label='BSADF episode (PSY)')
    axes[1].plot(idx, o['res']['bsadf'], color=IDAred, lw=0.8, label='BSADF (PSY)')
    axes[1].plot(idx, o['cv']['bsadf95'], color=Forest, lw=1.0, ls='--', label='95% critical value of BSADF')
    axes[1].plot(idx, o['res']['fwd'], color=Purple, lw=0.8, label='Recursive ADF, start fixed (PWY)')
    axes[1].plot(idx, o['cv']['fwd95'], color=Orange, lw=1.0, ls='-.', label='95% critical value of the recursive ADF')
    axes[1].set_ylabel('Statistic')
    axes[0].set_title('S&P 500, monthly 1990-2026: PSY and PWY date-stamping', fontsize=9, loc='left')
    fig.tight_layout()
    fig_legend(fig, axes, ncol=2, y=0.0)
    save_fig('ch17_sem_b2')
    s = summarise(o, y)
    return dict(gsadf=s['gsadf'], sadf=s['sadf'], n_ep=len(s['episodes']), n_pwy=len(s['episodes_pwy']))


def fig_sem_size(keys, labs, name):
    """B9 / B10: marimea GSADF si rata episoadelor false (studiul de nivel din inference17.json)."""
    with open(os.path.join(HERE, 'inference17.json')) as f:
        sz = json.load(f)['size']
    x = np.arange(len(keys))
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 2.9))
    for ax, (a, b, ttl) in zip(axes, [('size_mc', 'size_wild', 'Rejection rate of GSADF at 5%'),
                                      ('fp_mc', 'fp_wild', 'Paths with a false date-stamped episode')]):
        v1 = [100 * sz['rows'][k][a] for k in keys]
        v2 = [100 * sz['rows'][k][b] for k in keys]
        ax.bar(x - 0.2, v1, 0.4, color=MainBlue, label='Monte Carlo critical values')
        ax.bar(x + 0.2, v2, 0.4, color=IDAred, label='Wild bootstrap critical values')
        for xi, a1, a2 in zip(x, v1, v2):
            ax.text(xi - 0.2, a1 + 1, f'{a1:.1f}', ha='center', fontsize=7.5, color='black')
            ax.text(xi + 0.2, a2 + 1, f'{a2:.1f}', ha='center', fontsize=7.5, color='black')
        if a == 'size_mc':
            ax.axhline(5, color=Gray, ls='--', lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(labs, fontsize=8)
        ax.set_ylabel('%')
        ax.set_ylim(0, 100 if a == 'fp_mc' else max(v1 + v2) * 1.25)
        ax.set_title(ttl, fontsize=9, loc='left')
    fig.tight_layout()
    fig_legend(fig, axes, ncol=2, y=0.0)
    save_fig(name)
    return {k: sz['rows'][k] for k in keys}

## Run

In [ ]:
for f in [a1_present_value, a2_blanchard_watson, a3_df_limit, a4_mild, a4_windows, a5_markov, a6_markov, a7_lppls, a8_lppls]:
    print(f.__name__, f())
print('B1', b1_pd_windows())
print('B2', b2_sadf_vs_gsadf())
print('B3', b3_btc_wild())
print('B4', b4_bvb())
print('B5', b5_tc_windows())
print('B6', b6_ci_ndx())
print('B7', b7_ms_sp())
print('B8', b8_ms_btc())
print('C1', c1_ai())
# charts for the tasks and solutions of the seminar slides
print(fig_sem_data())
for f in [fig_sem_a1, fig_sem_a2, fig_sem_a3, fig_sem_a4, fig_sem_a5, fig_sem_a6, fig_sem_a7, fig_sem_a8]:
    print(f.__name__, f())
print(fig_sem_b1())
print(fig_sem_b2())
print(fig_sem_size(['iid', 'up'], ['Constant volatility', 'Volatility x3 at T/2'], 'ch17_sem_b9'))
print(fig_sem_size(['garch', 'down'], ['GARCH(1,1)', 'Volatility x1/3 at T/2'], 'ch17_sem_b10'))

![ch17_sem_data](./ch17_sem_data.png)

Output: `ch17_sem_data.pdf`

![ch17_sem_a1](./ch17_sem_a1.png)

Output: `ch17_sem_a1.pdf`

![ch17_sem_a2](./ch17_sem_a2.png)

Output: `ch17_sem_a2.pdf`

![ch17_sem_a3](./ch17_sem_a3.png)

Output: `ch17_sem_a3.pdf`

![ch17_sem_a4](./ch17_sem_a4.png)

Output: `ch17_sem_a4.pdf`

![ch17_sem_a5](./ch17_sem_a5.png)

Output: `ch17_sem_a5.pdf`

![ch17_sem_a6](./ch17_sem_a6.png)

Output: `ch17_sem_a6.pdf`

![ch17_sem_a7](./ch17_sem_a7.png)

Output: `ch17_sem_a7.pdf`

![ch17_sem_a8](./ch17_sem_a8.png)

Output: `ch17_sem_a8.pdf`

![ch17_sem_b1](./ch17_sem_b1.png)

Output: `ch17_sem_b1.pdf`

![ch17_sem_b2](./ch17_sem_b2.png)

Output: `ch17_sem_b2.pdf`

![ch17_sem_btc_wild](./ch17_sem_btc_wild.png)

Output: `ch17_sem_btc_wild.pdf`

![ch17_sem_bvb](./ch17_sem_bvb.png)

Output: `ch17_sem_bvb.pdf`

![ch17_sem_tc_windows](./ch17_sem_tc_windows.png)

Output: `ch17_sem_tc_windows.pdf`

![ch17_sem_ci_ndx](./ch17_sem_ci_ndx.png)

Output: `ch17_sem_ci_ndx.pdf`

![ch17_sem_ms_sp](./ch17_sem_ms_sp.png)

Output: `ch17_sem_ms_sp.pdf`

![ch17_sem_ms_btc3](./ch17_sem_ms_btc3.png)

Output: `ch17_sem_ms_btc3.pdf`

![ch17_sem_b9](./ch17_sem_b9.png)

Output: `ch17_sem_b9.pdf`

![ch17_sem_b10](./ch17_sem_b10.png)

Output: `ch17_sem_b10.pdf`

![ch17_sem_ai](./ch17_sem_ai.png)

Output: `ch17_sem_ai.pdf`